In [1]:
# ============================================================
# Vesuvius nnUNet MULTI-MODE ENSEMBLE (modes 1–10 registry)
# 2-GPU PARALLEL BATCHING + DISK-SAFE (/tmp) + DEADLOCK-SAFE LOG STREAMING
#
# BASE:  LB 0.613 "nnUNet MULTI-MODEL chunk ensemble" notebook
#
# CHANGES (per your request):
#   - ACTIVE ensemble: (7 + 5 + 2 + 1)
#   - DIRECT_WEIGHTED weights: M7=0.42, M5=0.18, M2=0.28, M1=0.12 (sums to 1)
#   - Threshold: PROB_T = 0.26
#   - Postprocess CHANGED:
#        from: Patch+Plug+Fill + binary_closing(5)
#        to:   closing(7,7) + check holes (PatchAndPlugAndFillCavities-style)
#
# Peak-disk reduction controls preserved:
# ✅ Chunked test processing (CASES_PER_CHUNK)
# ✅ For each chunk: predict -> per-case load prob -> float16-quantize -> fuse+PP -> write->zip->delete
# ✅ No "store all cases float16 maps" phase
# ✅ No "store all TIFFs then zip" phase
# ============================================================

import os, json, shutil, subprocess, re, zipfile, threading
from pathlib import Path
from typing import Optional, Union, List, Dict

import numpy as np
import tifffile
from tqdm.auto import tqdm

# ----------------------------
# Install (offline) - no pip cache
# ----------------------------
!pip install -q --no-index --no-cache-dir -f /kaggle/input/surface-packages-offline nnunetv2

# ============================================================
# Debug / logging options
# ============================================================
DEBUG_DISK_REPORT = False
DISK_REPORT_INCLUDE_WORKING = True

# ============================================================
# Peak-disk reduction controls
# ============================================================
PROCESS_IN_CHUNKS = True
CASES_PER_CHUNK   = 32
ZIP_TIFF_IMMEDIATELY = True
EMULATE_FLOAT16_MAPS = True  # keep True to match previous behavior

# ============================================================
# Paths / Settings
# ============================================================
INPUT_DIR   = Path("/kaggle/input/vesuvius-challenge-surface-detection")

# /tmp scratch for ALL big intermediates
WORKING_DIR = Path("/tmp/vesuvius_multi_scratch")
OUTPUT_DIR  = Path("/kaggle/working")

WORKING_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Cache / temp redirection ----
CACHE_ROOT = WORKING_DIR / "_cache"
TMP_ROOT   = WORKING_DIR / "_tmp"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TMP_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["XDG_CACHE_HOME"]  = str(CACHE_ROOT / "xdg")
os.environ["PIP_CACHE_DIR"]   = str(CACHE_ROOT / "pip")
os.environ["NUMBA_CACHE_DIR"] = str(CACHE_ROOT / "numba")
os.environ["CUPY_CACHE_DIR"]  = str(CACHE_ROOT / "cupy")
os.environ["TORCH_HOME"]      = str(CACHE_ROOT / "torch")
os.environ["MPLCONFIGDIR"]    = str(CACHE_ROOT / "mpl")

os.environ["TMPDIR"] = str(TMP_ROOT)
os.environ["TEMP"]   = str(TMP_ROOT)
os.environ["TMP"]    = str(TMP_ROOT)

os.environ["PIP_NO_CACHE_DIR"] = "1"

# nnUNet knobs
os.environ["nnUNet_USE_BLOSC2"] = "1"
os.environ["nnUNet_compile"] = "false"

# ---- Per-run cleanup toggles ----
CLEAN_RUN_CACHES = True
CLEAN_NNUNET_PREPROCESSED = True
CLEAN_NNUNET_RESULTS = True

FOLD: Union[int, str] = "all"
CONFIGURATION = "3d_fullres"

# ============================================================
# ACTIVE MODES + FUSION (UPDATED)
# ============================================================
ACTIVE_MODES = [7, 5, 2, 1]
FUSION_SCHEME = "DIRECT_WEIGHTED"
PROB_T = 0.26

# Requested weights (sum to 1)
W1  = 0.12
W2  = 0.28
W3  = 0.0
W4  = 0.0
W5  = 0.18
W6  = 0.0
W7  = 0.42
W8  = 0.0
W9  = 0.0
W10 = 0.0
WEIGHTS: Dict[int, float] = {1: W1, 2: W2, 3: W3, 4: W4, 5: W5, 6: W6, 7: W7, 8: W8, 9: W9, 10: W10}

# Keep these defined for compatibility (not used when DIRECT_WEIGHTED)
P54_W5 = 0.7
P54_W4 = 0.3
P21_W2 = 0.7
P21_W1 = 0.3
FINAL_W54 = 0.6
FINAL_W21 = 0.4

# Parallel CPU workers per process
NPP = 1
NPS = 1

# ============================================================
# nnUNet env roots (keep in /tmp)
# ============================================================
NNUNET_BASE         = WORKING_DIR / "nnUNet_data"
NNUNET_RAW          = NNUNET_BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = NNUNET_BASE / "nnUNet_preprocessed"
NNUNET_RESULTS      = WORKING_DIR / "nnUNet_results"

TEST_INPUT_DIR = WORKING_DIR / "test_input"
FINAL_TIFF_DIR = WORKING_DIR / "predictions_tiff"
SUB_ZIP        = OUTPUT_DIR / "submission.zip"

NNUNET_OUT: Dict[int, Path] = {m: WORKING_DIR / f"nnunet_out_m{m}" for m in range(1, 11)}

# ============================================================
# Utilities (disk reporting + safe IO)
# ============================================================
def _sh(cmd: str) -> str:
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    return out.strip()

def disk_report(tag: str):
    if not DEBUG_DISK_REPORT:
        return
    print(f"\n==================== DISK REPORT: {tag} ====================")
    try:
        du = shutil.disk_usage("/tmp")
        print(f"/tmp disk_usage: total={du.total/1024**3:.2f}G used={du.used/1024**3:.2f}G free={du.free/1024**3:.2f}G")
    except Exception as e:
        print("disk_usage(/tmp) failed:", repr(e))

    if DISK_REPORT_INCLUDE_WORKING:
        try:
            duw = shutil.disk_usage(str(OUTPUT_DIR))
            print(f"{OUTPUT_DIR} disk_usage: total={duw.total/1024**3:.2f}G used={duw.used/1024**3:.2f}G free={duw.free/1024**3:.2f}G")
        except Exception as e:
            print(f"disk_usage({OUTPUT_DIR}) failed:", repr(e))

    print(_sh("df -h / /tmp || true"))
    if DISK_REPORT_INCLUDE_WORKING:
        print(_sh(f"df -h {OUTPUT_DIR} || true"))

    print(_sh(f"du -sh {WORKING_DIR} 2>/dev/null || true"))
    print(_sh(f"du -sh {NNUNET_PREPROCESSED} 2>/dev/null || true"))
    print(_sh(f"du -sh {NNUNET_RESULTS} 2>/dev/null || true"))
    print(_sh(f"du -sh {CACHE_ROOT} 2>/dev/null || true"))
    if DISK_REPORT_INCLUDE_WORKING:
        print(_sh(f"du -sh {OUTPUT_DIR} 2>/dev/null || true"))
        print(_sh(f"ls -lh {OUTPUT_DIR} | head -n 50 || true"))
    print("============================================================\n")

def safe_rmtree(p: Path):
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)

def safe_mkdir_clean(p: Path):
    if p.exists():
        safe_rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def popen(cmd: str, env: Optional[dict] = None) -> subprocess.Popen:
    return subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=env,
        cwd=str(WORKING_DIR),
    )

def run(cmd: str, env: Optional[dict] = None, timeout: Optional[int] = None) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env, timeout=timeout, cwd=str(WORKING_DIR))

def stream_process_output_async(p: subprocess.Popen, prefix: str) -> threading.Thread:
    assert p.stdout is not None
    def _worker():
        for line in p.stdout:
            print(f"[{prefix}] {line.rstrip()}")
    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    return t

def get_gpu_ids() -> List[str]:
    try:
        import torch
        n = int(torch.cuda.device_count())
    except Exception:
        n = 0
    if n >= 2:
        return ["0", "1"]
    if n == 1:
        return ["0"]
    return []

def setup_environment():
    for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, OUTPUT_DIR, TEST_INPUT_DIR, FINAL_TIFF_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    os.environ["nnUNet_raw"] = str(NNUNET_RAW)
    os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
    os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

    print("WORKING_DIR (/tmp scratch):", WORKING_DIR)
    print("OUTPUT_DIR  (/kaggle/working):", OUTPUT_DIR)
    print("nnUNet_raw:", NNUNET_RAW)
    print("nnUNet_preprocessed:", NNUNET_PREPROCESSED)
    print("nnUNet_results:", NNUNET_RESULTS)
    print("CACHE_ROOT:", CACHE_ROOT)
    print("TMP_ROOT:", TMP_ROOT)
    print("nnUNet_USE_BLOSC2:", os.environ.get("nnUNet_USE_BLOSC2"))
    print("CUDA_VISIBLE_DEVICES (parent):", os.environ.get("CUDA_VISIBLE_DEVICES", "ALL"))

def chunked(seq: List[Path], k: int) -> List[List[Path]]:
    return [seq[i:i+k] for i in range(0, len(seq), k)]

# ============================================================
# MODELS[1..10] registry
# ============================================================
def _resolve_ckpt(meta_src: Path, ckpt_name: Optional[str] = None) -> Path:
    meta_src = Path(meta_src)
    candidates: List[Path] = []
    fold_all = meta_src / "fold_all"

    if ckpt_name:
        candidates += [fold_all / ckpt_name, meta_src / ckpt_name]
    else:
        for n in ("checkpoint_final.pth", "checkpoint_best.pth"):
            candidates += [fold_all / n, meta_src / n]

    for p in candidates:
        if p.exists():
            return p

    hits = sorted(meta_src.rglob("checkpoint_*.pth"))
    if hits:
        for p in hits:
            if p.name == "checkpoint_final.pth":
                return p
        return hits[0]

    raise FileNotFoundError(f"No checkpoint found under {meta_src} (ckpt_name={ckpt_name})")

def make_model(mode: int, tag: str, plans: str, trainer: str, meta_src: str,
               ckpt: Optional[str] = None, ckpt_name: Optional[str] = None) -> dict:
    ds_id = 100 + mode
    ds_name = f"Dataset{ds_id:03d}_VesuviusSurface_M{mode}"
    meta_src_p = Path(meta_src)
    ckpt_p = Path(ckpt) if ckpt is not None else _resolve_ckpt(meta_src_p, ckpt_name=ckpt_name)
    return dict(mode=mode, name=f"M{mode}_{tag}", ds_id=ds_id, ds_name=ds_name,
                plans=plans, trainer=trainer, ckpt=ckpt_p, meta_src=meta_src_p)

MODELS: Dict[int, dict] = {
    1: make_model(
        mode=1, tag="MPlans_4000_patch128",
        plans="nnUNetResEncUNetMPlans", trainer="nnUNetTrainer_4000epochs",
        meta_src="/kaggle/input/nnunet-ep4000-all",
        ckpt="/kaggle/input/nnunet-ep4000-all/checkpoint_final.pth",
    ),
    2: make_model(
        mode=2, tag="XLPlans_250_patch192_best",
        plans="nnUNetResEncUNetXLPlans", trainer="nnUNetTrainer_250epochs",
        meta_src="/kaggle/input/kaggle-vesuvius-dataset/nnUNetTrainer_250epochs__nnUNetResEncUNetXLPlans__3d_fullres",
        ckpt_name="checkpoint_best.pth",
    ),
    3: make_model(
        mode=3, tag="XLPlans_250_patch192_final",
        plans="nnUNetResEncUNetXLPlans", trainer="nnUNetTrainer_250epochs",
        meta_src="/kaggle/input/kaggle-vesuvius-dataset/nnUNetTrainer_250epochs__nnUNetResEncUNetXLPlans__3d_fullres",
        ckpt_name="checkpoint_final.pth",
    ),
    4: make_model(
        mode=4, tag="XLPlans_250_patch256_best",
        plans="nnUNetResEncUNetXLPlans", trainer="nnUNetTrainer_250epochs",
        meta_src="/kaggle/input/kaggle-vesuvius-dataset-nnunet-256",
        ckpt_name="checkpoint_best.pth",
    ),
    5: make_model(
        mode=5, tag="XLPlans_250_patch256_final",
        plans="nnUNetResEncUNetXLPlans", trainer="nnUNetTrainer_250epochs",
        meta_src="/kaggle/input/kaggle-vesuvius-dataset-nnunet-256",
        ckpt_name="checkpoint_final.pth",
    ),

    # ---- ADDED 6–10 (as provided) ----
    6: make_model(
        mode=6,
        tag="06_XLPlans_500_patch288_E478",
        plans="nnUNetResEncUNetXLPlans",
        trainer="nnUNetTrainer_500epochs",
        meta_src="/kaggle/input/datasets/yiheng/kaggle-vesuvius-dataset-nnunet-288",
        ckpt_name="checkpoint_478e.pth",
    ),
    7: make_model(
        mode=7,
        tag="07_LPlans_4000_patch192_fromscratch_final",
        plans="nnUNetResEncUNetLPlans",
        trainer="nnUNetTrainer_4000epochs",
        meta_src="/kaggle/input/datasets/yiheng/kaggle-vesuvius-dataset-nnunet-192-from-scratch",
        ckpt_name="checkpoint_final.pth",
    ),
    8: make_model(
        mode=8,
        tag="8_LPlans_4000_patch224_fromscratch_final",
        plans="nnUNetResEncUNetLPlans",
        trainer="nnUNetTrainer_4000epochs",
        meta_src="/kaggle/input/datasets/yiheng/kaggle-vesuvius-dataset-nnunet-224-from-scratch",
        ckpt_name="checkpoint_best.pth",
    ),
    9: make_model(
        mode=9,
        tag="09_LPlans_4000_patch160_fromscratch_best",
        plans="nnUNetResEncUNetLPlans",
        trainer="nnUNetTrainer_4000epochs",
        meta_src="/kaggle/input/datasets/yiheng/kaggle-vesuvius-dataset-nnunet-160-from-scratch",
        ckpt_name="checkpoint_best.pth",
    ),
    10: make_model(
        mode=10,
        tag="10_LPlans_4000_patch224_fromscratch_final",
        plans="nnUNetResEncUNetLPlans",
        trainer="nnUNetTrainer_4000epochs",
        meta_src="/kaggle/input/datasets/yiheng/kaggle-vesuvius-dataset-nnunet-224-from-scratch",
        ckpt_name="checkpoint_final.pth",
    ),
}

def ensure_model_registered(model: dict) -> Path:
    ds_name = model["ds_name"]
    ds_id   = int(model["ds_id"])
    if not ds_name.startswith(f"Dataset{ds_id:03d}_"):
        raise ValueError(f"ds_name must start with Dataset{ds_id:03d}_ ; got {ds_name}")

    trainer = model["trainer"]
    plans   = model["plans"]

    model_root = NNUNET_RESULTS / ds_name / f"{trainer}__{plans}__{CONFIGURATION}"
    fold_dir   = model_root / "fold_all"
    fold_dir.mkdir(parents=True, exist_ok=True)

    ckpt_src = Path(model["ckpt"])
    if not ckpt_src.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_src}")

    ckpt_dst = fold_dir / "checkpoint_final.pth"
    if ckpt_dst.exists() or ckpt_dst.is_symlink():
        ckpt_dst.unlink()
    ckpt_dst.symlink_to(ckpt_src)

    meta_src = Path(model["meta_src"])
    for fname in ["dataset.json", "plans.json", "dataset_fingerprint.json"]:
        src = meta_src / fname
        dst = model_root / fname
        if not src.exists():
            raise FileNotFoundError(f"Missing {fname} at {src}")
        if dst.exists():
            dst.unlink()
        shutil.copy2(src, dst)

    resolved = ckpt_dst.resolve()
    print(f"Registered {model['name']}")
    print("  model_root:", model_root)
    print("  ckpt_src :", ckpt_src, f"(size={ckpt_src.stat().st_size/1024/1024:.1f} MB)")
    print("  ckpt_link:", ckpt_dst, "->", resolved)
    assert resolved == ckpt_src.resolve(), f"Symlink sanity failed: {resolved} != {ckpt_src.resolve()}"
    return model_root

# ============================================================
# Test data prep (subset-aware)
# ============================================================
def create_spacing_json(path: Path, spacing=(1.0, 1.0, 1.0)):
    with open(path, "w") as f:
        json.dump({"spacing": list(spacing)}, f)

def list_all_test_tifs() -> List[Path]:
    test_images_dir = INPUT_DIR / "test_images"
    tifs = sorted(test_images_dir.glob("*.tif"))
    print("Found test cases:", len(tifs))
    if not tifs:
        raise RuntimeError(f"No test tif found under {test_images_dir}")
    return tifs

def prepare_test_data_subset(tifs: List[Path]) -> List[str]:
    safe_mkdir_clean(TEST_INPUT_DIR)
    for p in tqdm(tifs, desc="Preparing test data (chunk)"):
        case_id = p.stem
        img_dst = TEST_INPUT_DIR / f"{case_id}_0000.tif"
        if img_dst.exists() or img_dst.is_symlink():
            img_dst.unlink()
        img_dst.symlink_to(p.resolve())
        create_spacing_json(TEST_INPUT_DIR / f"{case_id}_0000.json")
        create_spacing_json(TEST_INPUT_DIR / f"{case_id}.json")
    return [p.stem for p in tifs]

# ============================================================
# nnUNet predict helpers
# ============================================================
def detect_save_prob_flag() -> str:
    help_out = run("nnUNetv2_predict -h")
    txt = (help_out.stdout or "") + "\n" + (help_out.stderr or "")

    candidates = [
        "--save_probabilities",
        "--save_probabilities_in_separate_folder",
        "--save_probabilities_in_npz",
        "--export_probabilities",
    ]
    for c in candidates:
        if c in txt:
            print("Detected probability flag:", c)
            return c

    m = re.search(r"--save[_-]prob\w+", txt)
    if m:
        print("Detected probability flag (regex):", m.group(0))
        return m.group(0)

    print("nnUNetv2_predict -h tail:\n", txt[-2500:])
    raise RuntimeError("Could not find probability-saving flag in nnUNetv2_predict help.")

def build_predict_cmd(model: dict, out_dir: Path, save_prob_flag: str) -> str:
    return (
        f"nnUNetv2_predict "
        f"-d {int(model['ds_id']):03d} -c {CONFIGURATION} -f {FOLD} "
        f"-i {TEST_INPUT_DIR} -o {out_dir} "
        f"-p {model['plans']} -tr {model['trainer']} "
        f"-npp {NPP} -nps {NPS} --verbose "
        f"{save_prob_flag}"
    )

def run_two_models_parallel(model1: dict, out1: Path, gpu1: str,
                            model2: dict, out2: Path, gpu2: str,
                            save_prob_flag: str,
                            stage: int):
    disk_report(f"BEFORE Stage {stage} (parallel)")

    cmd1 = build_predict_cmd(model1, out1, save_prob_flag)
    cmd2 = build_predict_cmd(model2, out2, save_prob_flag)

    env1 = os.environ.copy(); env1["CUDA_VISIBLE_DEVICES"] = gpu1
    env2 = os.environ.copy(); env2["CUDA_VISIBLE_DEVICES"] = gpu2

    print(f"\n=== Stage {stage}: parallel inference ===")
    print(f"GPU{gpu1}:", cmd1)
    print(f"GPU{gpu2}:", cmd2)

    p1 = popen(cmd1, env=env1)
    p2 = popen(cmd2, env=env2)

    t1 = stream_process_output_async(p1, model1["name"])
    t2 = stream_process_output_async(p2, model2["name"])

    rc1 = p1.wait()
    rc2 = p2.wait()
    t1.join(timeout=2)
    t2.join(timeout=2)

    if rc1 != 0 or rc2 != 0:
        disk_report(f"AFTER Stage {stage} (parallel) - FAILED")
        raise RuntimeError(f"nnUNetv2_predict failed: rc1={rc1}, rc2={rc2}")

    disk_report(f"AFTER Stage {stage} (parallel)")

def run_one_model(model: dict, out_dir: Path, gpu: str, save_prob_flag: str, stage: int):
    disk_report(f"BEFORE Stage {stage} (single)")

    cmd = build_predict_cmd(model, out_dir, save_prob_flag)
    env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = gpu

    print(f"\n=== Stage {stage}: single inference ===")
    print(f"GPU{gpu}:", cmd)

    p = popen(cmd, env=env)
    t = stream_process_output_async(p, model["name"])
    rc = p.wait()
    t.join(timeout=2)

    if rc != 0:
        disk_report(f"AFTER Stage {stage} (single) - FAILED")
        raise RuntimeError(f"nnUNetv2_predict failed for {model['name']}: rc={rc}")

    disk_report(f"AFTER Stage {stage} (single)")

def run_active_models_in_stages(active_modes: List[int], save_prob_flag: str):
    gpu_ids = get_gpu_ids()
    if not gpu_ids:
        raise RuntimeError("No CUDA GPU detected. This pipeline expects at least 1 GPU.")

    modes = list(active_modes)
    stage = 1
    i = 0
    while i < len(modes):
        if len(gpu_ids) >= 2 and (i + 1) < len(modes):
            m1, m2 = modes[i], modes[i + 1]
            run_two_models_parallel(MODELS[m1], NNUNET_OUT[m1], gpu_ids[0],
                                    MODELS[m2], NNUNET_OUT[m2], gpu_ids[1],
                                    save_prob_flag, stage=stage)
            i += 2
        else:
            m1 = modes[i]
            run_one_model(MODELS[m1], NNUNET_OUT[m1], gpu_ids[0], save_prob_flag, stage=stage)
            i += 1
        stage += 1

# ============================================================
# Load class-1 prob DIRECTLY from nnUNet npz (per-case)
# ============================================================
def find_case_npz(pred_dir: Path, case_id: str) -> Path:
    direct = pred_dir / f"{case_id}.npz"
    if direct.exists():
        return direct

    probs = pred_dir / "probabilities" / f"{case_id}.npz"
    if probs.exists():
        return probs

    alt = pred_dir / f"{case_id}_0000.npz"
    if alt.exists():
        return alt

    probs_alt = pred_dir / "probabilities" / f"{case_id}_0000.npz"
    if probs_alt.exists():
        return probs_alt

    hits = sorted(pred_dir.rglob(f"{case_id}*.npz"))
    if hits:
        return hits[0]

    raise FileNotFoundError(f"No NPZ found for {case_id} under {pred_dir}")

def load_surface_prob_from_npz(npz_path: Path) -> np.ndarray:
    with np.load(npz_path) as data:
        keys = list(data.keys())
        arr = None
        for k in ["probabilities", "softmax", "predicted_probabilities", "probs"]:
            if k in data:
                arr = data[k]
                break
        if arr is None:
            arr = data[keys[0]]

    arr = np.asarray(arr)
    if arr.ndim != 4:
        raise ValueError(f"Unexpected prob tensor shape {arr.shape} in {npz_path}")

    # try to infer channel axis
    if arr.shape[0] <= 10 and arr.shape[1] > 10:
        p = arr[1].astype(np.float32)
    elif arr.shape[-1] <= 10 and arr.shape[0] > 10:
        p = arr[..., 1].astype(np.float32)
    else:
        p = arr[1].astype(np.float32)

    if EMULATE_FLOAT16_MAPS:
        p = p.astype(np.float16).astype(np.float32)

    return p

def load_p_from_nnunet(mode: int, cid: str) -> np.ndarray:
    npz = find_case_npz(NNUNET_OUT[mode], cid)
    return load_surface_prob_from_npz(npz)

# ============================================================
# Fusion helpers
# ============================================================
def normalize_weights(active_modes: List[int], weights: Dict[int, float]) -> Dict[int, float]:
    w = {m: float(weights.get(m, 0.0)) for m in active_modes}
    w = {m: v for m, v in w.items() if v > 0}
    s = sum(w.values())
    if s <= 0:
        raise ValueError(f"All weights are zero/non-positive for ACTIVE_MODES={active_modes}.")
    return {m: v / s for m, v in w.items()}

def fuse_probability_from_nnunet(cid: str, active_modes: List[int]) -> np.ndarray:
    if FUSION_SCHEME == "DIRECT_WEIGHTED":
        w_norm = normalize_weights(active_modes, WEIGHTS)
        p = None
        for m, w in w_norm.items():
            pm = load_p_from_nnunet(m, cid)
            if p is None:
                p = w * pm
            else:
                p += w * pm
        assert p is not None
        return p

    if FUSION_SCHEME == "PAIR_ENSEMBLE":
        required = {1, 2, 4, 5}
        if not required.issubset(set(active_modes)):
            raise ValueError(f"PAIR_ENSEMBLE requires ACTIVE_MODES to include {sorted(required)}; got {active_modes}")

        p5 = load_p_from_nnunet(5, cid)
        p4 = load_p_from_nnunet(4, cid)
        p2 = load_p_from_nnunet(2, cid)
        p1 = load_p_from_nnunet(1, cid)

        p54 = (P54_W5 * p5) + (P54_W4 * p4)
        p21 = (P21_W2 * p2) + (P21_W1 * p1)
        p   = (FINAL_W54 * p54) + (FINAL_W21 * p21)
        return p

    raise ValueError(f"Unknown FUSION_SCHEME={FUSION_SCHEME}")

# ============================================================
# Postprocess (REPLACED): closing(7,7) + check holes (PatchAndPlugAndFillCavities-style)
# ============================================================
from numba import jit
from skimage import morphology as sk_morphology
import cv2
import cupy
import cupyx
from cupyx.scipy import ndimage  # kept

EULER_COEFS3D_26 = cupy.array([
    0, 1, 1, 0, 1, 0, -2, -1, 1, -2, 0, -1, 0, -1, -1, 0, 1, 0, -2, -1, -2, -1, -1, -2, -6, -3, -3, -2, -3, -2, 0, -1, 1, -2,
    0, -1, -6, -3, -3, -2, -2, -1, -1, -2, -3, 0, -2, -1, 0, -1, -1, 0, -3, -2, 0, -1, -3, 0, -2, -1, 0, 1, 1, 0, 1, -2, -6, -3, 0, -1, -3, -2, -2, -1,
    -3, 0, -1, -2, -2, -1, 0, -1, -3, -2, -1, 0, 0, -1, -3, 0, 0, 1, -2, -1, 1, 0, -2, -1, -3, 0, -3, 0, 0, 1, -1, 4, 0, 3, 0, 3, 1, 2, -1, -2, -2, -1,
    -2, -1, 1, 0, 0, 3, 1, 2, 1, 2, 2, 1, 1, -6, -2, -3, -2, -3, -1, 0, 0, -3, -1, -2, -1, -2, -2, -1, -2, -3, -1, 0, -1, 0, 4, 3, -3, 0, 0, 1, 0, 1,
    3, 2, 0, -3, -1, -2, -3, 0, 0, 1, -1, 0, 0, -1, -2, 1, -1, 0, -1, -2, -2, -1, 0, 1, 3, 2, -2, 1, -1, 0, 1, 2, 2, 1, 0, -3, -3, 0, -1, -2, 0, 1,
    -1, 0, -2, 1, 0, -1, -1, 0, -1, -2, 0, 1, -2, -1, 3, 2, -2, 1, 1, 2, -1, 0, 2, 1, -1, 0, -2, 1, -2, 1, 1, 2, -2, 3, -1, 2, -1, 2, 0, 1, 0, -1,
    -1, 0, -1, 0, 2, 1, -1, 2, 0, 1, 0, 1, 1, 0
], dtype=cupy.int32)

def cupy_euler_number(image, connectivity: int):
    image = cupy.asarray(image)
    image = (image > 0).astype(int)
    image = cupy.pad(image, pad_width=1, mode='constant')

    if image.ndim != 3:
        raise ValueError('Input image is not 3D.')

    if connectivity == 2:
        raise NotImplementedError('For 3D images, Euler number is implemented for connectivities 1 and 3 only')

    config = cupy.array([
        [[0, 0, 0], [0, 0, 0], [0, 0, 0]],
        [[0, 0, 0], [0, 1, 4], [0, 2, 8]],
        [[0, 0, 0], [0, 16, 64], [0, 32, 128]],
    ])

    if connectivity == 1:
        coefs = EULER_COEFS3D_26[::-1]
    else:
        coefs = EULER_COEFS3D_26

    XF = cupyx.scipy.ndimage.convolve(image, config, mode='constant', cval=0)
    h = cupy.bincount(XF.ravel(), minlength=256)
    return int((0.125 * coefs @ h).item())

def get_best_projection(sheet_vol):
    best_axis, best_area, best_mask = 0, 0, None
    for axis in range(3):
        proj_mask = cupy.max(sheet_vol, axis)
        proj_area = cupy.sum(proj_mask)
        if proj_area > best_area:
            best_axis = axis
            best_area = proj_area
            best_mask = proj_mask
    return best_axis, best_mask

def get_hole_patches(sheet_vol, proj_axis, proj_mask, params):
    proj_mask = cupy.asnumpy(proj_mask)
    heightmap = cupy.asnumpy(cupy.argmax(sheet_vol, proj_axis))

    bdr = params['border_width']
    if bdr > 0:
        heightmap = heightmap[bdr:-bdr, bdr:-bdr]
        proj_mask = proj_mask[bdr:-bdr, bdr:-bdr]

    num_holes, hole_labels_img, hole_stats, _ = (
        cv2.connectedComponentsWithStats(1 - proj_mask, connectivity=8, ltype=cv2.CV_16U)
    )

    patches = []
    for i in range(1, num_holes):
        x0, y0, hole_w, hole_h, _ = hole_stats[i]

        if (x0 == 0) or (y0 == 0) or (y0 + hole_h >= heightmap.shape[0]) or (x0 + hole_w >= heightmap.shape[1]):
            continue

        ph, pw = hole_h + 2, hole_w + 2
        interp_hm_h = np.full((ph, pw), -1, dtype=np.float32)
        interp_hm_v = np.full((ph, pw), -1, dtype=np.float32)
        wts_h = np.full((ph, pw), 0, dtype=np.float32)
        wts_v = np.full((ph, pw), 0, dtype=np.float32)

        hole_mask = (hole_labels_img[y0-1:y0+ph-1, x0-1:x0+pw-1] == i)
        not_hole_mask = ~hole_mask

        # horizontal interpolation
        for y in range(0, ph):
            hm_row = heightmap[y + y0 - 1, x0 - 1:]
            mask_row = hole_mask[y]
            wts_h_row = wts_h[y]
            interp_hm_h_row = interp_hm_h[y]
            ramping = False
            x = 0
            while x < pw:
                if not ramping:
                    if (not mask_row[x]) and (x < pw - 1) and mask_row[x + 1]:
                        ramping = True
                        ramp_start = (x, hm_row[x])
                else:
                    if not mask_row[x]:
                        ramping = False
                        ramp_end = (x, hm_row[x])
                        slope = (ramp_end[1] - ramp_start[1]) / (ramp_end[0] - ramp_start[0])
                        ht = ramp_start[1] + slope
                        for xx in range(ramp_start[0] + 1, ramp_end[0]):
                            interp_hm_h_row[xx] = ht
                            ht += slope
                            wts_h_row[xx] = 1 / min(xx - ramp_start[0], ramp_end[0] - xx)
                        x -= 1
                x += 1

        # vertical interpolation
        for x in range(0, pw):
            hm_col = heightmap[y0 - 1:, x + x0 - 1]
            mask_col = hole_mask[:, x]
            wts_v_col = wts_v[:, x]
            interp_hm_v_col = interp_hm_v[:, x]
            ramping = False
            y = 0
            while y < ph:
                if not ramping:
                    if (not mask_col[y]) and (y < ph - 1) and mask_col[y + 1]:
                        ramping = True
                        ramp_start = (y, hm_col[y])
                else:
                    if not mask_col[y]:
                        ramping = False
                        ramp_end = (y, hm_col[y])
                        slope = (ramp_end[1] - ramp_start[1]) / (ramp_end[0] - ramp_start[0])
                        ht = ramp_start[1] + slope
                        for yy in range(ramp_start[0] + 1, ramp_end[0]):
                            interp_hm_v_col[yy] = ht
                            ht += slope
                            wts_v_col[yy] = 1 / min(yy - ramp_start[0], ramp_end[0] - yy)
                        y -= 1
                y += 1

        interp_hts = np.round(
            (wts_h * interp_hm_h + wts_v * interp_hm_v) / (wts_h + wts_v + 1e-8)
        ).astype(np.uint16)
        interp_hts[not_hole_mask] = heightmap[y0-1:y0-1+ph, x0-1:x0-1+pw][not_hole_mask]

        patch = np.zeros((ph, pw, 2), dtype=np.int32)
        for yy in range(1, hole_h + 1):
            for xx in range(1, hole_w + 1):
                nbr_hts = interp_hts[yy-1:yy+2, xx-1:xx+2]
                patch[yy, xx, 0] = np.min(nbr_hts)
                patch[yy, xx, 1] = np.max(nbr_hts)

        patch[:, :, 0][not_hole_mask] = -1
        patch[:, :, 1][not_hole_mask] = -1
        patch = np.ascontiguousarray(patch[1:-1, 1:-1, :])
        patches.append((patch, x0 + bdr, y0 + bdr, hole_w, hole_h))

    return patches, proj_axis

@jit(nopython=True)
def insert_patches_in_volume(patches, proj_axis, vol):
    pz_max = vol.shape[proj_axis]
    axes = (0,1,2) if proj_axis == 0 else (1,0,2) if proj_axis == 1 else (2,0,1)
    vol_t = np.transpose(vol, axes)

    for patch_info in patches:
        patch, px0, py0, pw, ph = patch_info
        for py in range(py0, py0+ph):
            for px in range(px0, px0+pw):
                for pz in range(patch[py-py0, px-px0, 0], patch[py-py0, px-px0, 1] + 1):
                    if (pz > 0) and (pz < pz_max):
                        vol_t[pz][py][px] = 1
    return vol

def plug_small_holes(in_vol, lut, params):
    dilation_struct = cupy.array(sk_morphology.footprint_rectangle((3,3,3)))
    dilated_vol = cupyx.scipy.ndimage.binary_dilation(in_vol, dilation_struct)

    voxel_locs = [cupy.asnumpy(a) for a in cupy.nonzero(dilated_vol)]
    out_vol = cupy.asnumpy(in_vol)
    count = plug_small_holes_numba(out_vol, voxel_locs, lut)
    count += plug_small_holes_numba(out_vol, voxel_locs, lut)
    return out_vol, count

@jit(nopython=True)
def plug_small_holes_numba(in_vol, voxel_locs, lut):
    count = 0
    for z, y, x in zip(voxel_locs[0], voxel_locs[1], voxel_locs[2]):
        lut_code = in_vol[z,y,x] +    2*in_vol[z,y,x+1] +    4*in_vol[z,y+1,x] +     8*in_vol[z,y+1,x+1] + \
                16*in_vol[z+1,y,x] + 32*in_vol[z+1,y,x+1] + 64*in_vol[z+1,y+1,x] + 128*in_vol[z+1,y+1,x+1]
        lut_val = lut[lut_code]
        if lut_val[0,0,0] < 256:
            in_vol[z:z+2, y:y+2, x:x+2] = lut_val
            count += 1
    return count

def code_to_array(code):
    result = np.zeros((8,), dtype=np.uint32)
    b = f"{code:08b}"
    for i in range(8):
        result[7-i] = int(b[i])
    return result.reshape((2,2,2))

def create_plug_lut():
    lut = [None] * 256
    for i in range(256):
        lut[i] = np.full((8,), 512, dtype=np.uint32).reshape((2,2,2))

    lut[6]   = code_to_array(7)
    lut[9]   = code_to_array(11)
    lut[18]  = code_to_array(19)
    lut[20]  = code_to_array(21)
    lut[22]  = code_to_array(23)
    lut[24]  = code_to_array(27)
    lut[25]  = code_to_array(27)
    lut[26]  = code_to_array(27)
    lut[28]  = code_to_array(29)
    lut[30]  = code_to_array(31)

    lut[33] = code_to_array(35)
    lut[36] = code_to_array(39)
    lut[37] = code_to_array(39)
    lut[38] = code_to_array(39)
    lut[40] = code_to_array(42)
    lut[41] = code_to_array(43)
    lut[44] = code_to_array(46)
    lut[45] = code_to_array(47)

    lut[52] = code_to_array(53)
    lut[54] = code_to_array(55)
    lut[56] = code_to_array(58)
    lut[57] = code_to_array(59)
    lut[60] = code_to_array(63)
    lut[61] = code_to_array(63)
    lut[62] = code_to_array(63)

    lut[65] = code_to_array(69)
    lut[66] = code_to_array(71)
    lut[67] = code_to_array(71)
    lut[70] = code_to_array(71)
    lut[72] = code_to_array(76)
    lut[73] = code_to_array(77)
    lut[74] = code_to_array(78)
    lut[75] = code_to_array(79)
    lut[82] = code_to_array(83)
    lut[86] = code_to_array(87)
    lut[88] = code_to_array(92)
    lut[89] = code_to_array(93)
    lut[90] = code_to_array(95)
    lut[91] = code_to_array(95)
    lut[94] = code_to_array(95)

    lut[96] = code_to_array(112)
    lut[97] = code_to_array(113)
    lut[98] = code_to_array(114)
    lut[99] = code_to_array(115)
    lut[100] = code_to_array(116)
    lut[101] = code_to_array(117)
    lut[102] = code_to_array(119)
    lut[103] = code_to_array(119)
    lut[104] = code_to_array(232)
    lut[105] = code_to_array(251)
    lut[106] = code_to_array(234)
    lut[108] = code_to_array(236)
    lut[107] = code_to_array(251)
    lut[109] = code_to_array(253)
    lut[110] = code_to_array(238)
    lut[111] = code_to_array(239)
    lut[118] = code_to_array(119)
    lut[120] = code_to_array(248)
    lut[121] = code_to_array(127)
    lut[122] = code_to_array(127)
    lut[123] = code_to_array(127)
    lut[124] = code_to_array(127)
    lut[125] = code_to_array(127)
    lut[126] = code_to_array(127)

    lut[129] = code_to_array(139)
    lut[130] = code_to_array(138)
    lut[131] = code_to_array(139)
    lut[132] = code_to_array(140)
    lut[133] = code_to_array(141)
    lut[134] = code_to_array(142)
    lut[135] = code_to_array(143)
    lut[137] = code_to_array(139)
    lut[144] = code_to_array(176)
    lut[145] = code_to_array(177)
    lut[146] = code_to_array(178)
    lut[147] = code_to_array(179)
    lut[148] = code_to_array(212)
    lut[149] = code_to_array(213)
    lut[150] = code_to_array(223)
    lut[151] = code_to_array(223)
    lut[152] = code_to_array(184)
    lut[153] = code_to_array(187)
    lut[154] = code_to_array(186)
    lut[155] = code_to_array(187)
    lut[156] = code_to_array(220)
    lut[157] = code_to_array(221)
    lut[158] = code_to_array(191)
    lut[159] = code_to_array(191)

    lut[161] = code_to_array(163)
    lut[164] = code_to_array(172)
    lut[165] = code_to_array(175)
    lut[166] = code_to_array(174)
    lut[167] = code_to_array(175)
    lut[169] = code_to_array(171)
    lut[173] = code_to_array(175)
    lut[180] = code_to_array(244)
    lut[181] = code_to_array(245)
    lut[182] = code_to_array(247)
    lut[183] = code_to_array(247)
    lut[185] = code_to_array(187)
    lut[188] = code_to_array(252)
    lut[189] = code_to_array(253)
    lut[190] = code_to_array(254)

    lut[193] = code_to_array(197)
    lut[194] = code_to_array(202)
    lut[195] = code_to_array(207)
    lut[198] = code_to_array(206)
    lut[199] = code_to_array(207)
    lut[201] = code_to_array(205)
    lut[203] = code_to_array(207)
    lut[210] = code_to_array(242)
    lut[211] = code_to_array(243)
    lut[214] = code_to_array(223)
    lut[215] = code_to_array(223)
    lut[217] = code_to_array(221)
    lut[218] = code_to_array(223)
    lut[219] = code_to_array(223)
    lut[222] = code_to_array(223)

    lut[225] = code_to_array(241)
    lut[227] = code_to_array(243)
    lut[229] = code_to_array(245)
    lut[230] = code_to_array(238)
    lut[231] = code_to_array(247)
    lut[233] = code_to_array(239)
    lut[235] = code_to_array(251)
    lut[237] = code_to_array(253)
    lut[246] = code_to_array(247)
    lut[249] = code_to_array(253)
    return lut

def postprocess(in_vol, params):
    # Zero border
    bw = params['border_width']
    if bw > 0:
        in_vol[0:bw, :, :] = in_vol[:, 0:bw, :] = in_vol[:, :, 0:bw] = 0
        in_vol[-bw:, :, :] = in_vol[:, -bw:, :] = in_vol[:, :, -bw:] = 0

    # Remove non-sheets
    pruned_vol = sk_morphology.remove_small_objects(in_vol.astype(bool), min_size=params['small_obj_thresh'])

    # Label sheets (26-connected)
    struct_element333 = cupy.array(sk_morphology.footprint_rectangle((3,3,3)))
    sheet_labels_vol, num_sheets = cupyx.scipy.ndimage.label(cupy.array(pruned_vol), struct_element333)

    out_vol = cupy.zeros(in_vol.shape, dtype=np.uint8)
    sheet_vol = cupy.empty(sheet_labels_vol.shape, dtype=np.uint8)

    lut = create_plug_lut()

    # Build oriented ellipsoidal closing elements
    close_par  = int(params.get('close_par', 7))
    close_perp = int(params.get('close_perp', 7))
    sz = close_par  # source notebook assumption; with 7/7 this is correct
    ctr = (sz - 1) // 2
    r_par  = (close_par  - 1) // 2
    r_perp = (close_perp - 1) // 2

    closing_element = np.zeros((sz, sz, sz), dtype=np.uint8)
    for z in range(sz):
        for y in range(sz):
            for x in range(sz):
                if ((z-ctr)/(r_perp+0.25))**2 + ((y-ctr)/(r_par+0.25))**2 + ((x-ctr)/(r_par+0.25))**2 <= 1:
                    closing_element[z, y, x] = 1

    closing_elements = []
    closing_elements.append(cupy.asarray(np.copy(np.transpose(closing_element, (0, 1, 2)))))
    closing_elements.append(cupy.asarray(np.copy(np.transpose(closing_element, (1, 0, 2)))))
    closing_elements.append(cupy.asarray(np.copy(np.transpose(closing_element, (1, 2, 0)))))

    max_patch_size = int(params.get('max_patch_size', 64))

    for sheet_idx in range(1, min(num_sheets, params['max_sheets']) + 1):
        # sheet mask
        sheet_vol.fill(0)
        sheet_vol[cupy.where(sheet_labels_vol == sheet_idx)] = 1

        # projection for orientation
        proj_axis, proj_mask = get_best_projection(sheet_vol)
        close_elem = closing_elements[proj_axis]

        # oriented closing, keep original too
        sheet_vol_closed = cupy.maximum(
            sheet_vol,
            cupyx.scipy.ndimage.binary_closing(sheet_vol, close_elem).astype(np.uint8)
        )

        # patches for holes (based on closed sheet)
        patches, _ = get_hole_patches(sheet_vol_closed, proj_axis, proj_mask, params)

        # apply only "small" patches
        small_patches = [p for p in patches if p[3] * p[4] < max_patch_size**2]
        sheet_vol_patched = sheet_vol_closed
        if len(small_patches) > 0:
            sheet_vol_patched = cupy.asarray(
                insert_patches_in_volume(small_patches, proj_axis, cupy.asnumpy(sheet_vol_closed))
            )
            sheet_vol_patched = cupy.maximum(
                sheet_vol_patched,
                cupyx.scipy.ndimage.binary_closing(sheet_vol_patched, close_elem).astype(np.uint8)
            )

        # plug small holes
        sheet_vol_fixed, _ = plug_small_holes(sheet_vol_patched, lut, params)
        sheet_vol_fixed = cupy.asarray(sheet_vol_fixed)

        # "check holes": discard patching if it made holes worse
        num_holes_after = 1 - cupy_euler_number(sheet_vol_fixed, connectivity=1)
        if num_holes_after > 0:
            sheet_vol_nopatch, _ = plug_small_holes(sheet_vol_closed, lut, params)
            sheet_vol_nopatch = cupy.asarray(sheet_vol_nopatch)
            num_holes_before = 1 - cupy_euler_number(sheet_vol_nopatch, connectivity=1)
            if num_holes_after > num_holes_before:
                sheet_vol_fixed = sheet_vol_nopatch

        occupied = cupyx.scipy.ndimage.binary_dilation((out_vol != 0), struct_element333)
        out_vol[cupy.where((sheet_vol_fixed != 0) & (occupied == 0))] = sheet_idx

    # fill cavities
    out_vol = cupyx.scipy.ndimage.binary_fill_holes(out_vol).astype(cupy.uint8)

    # zero border again
    bw = params['border_width']
    out_vol[0:bw,:,:] = out_vol[:,0:bw,:] = out_vol[:,:,0:bw] = 0
    out_vol[-bw:,:,:] = out_vol[:,-bw:,:] = out_vol[:,:,-bw:] = 0

    return cupy.asnumpy((out_vol != 0)).astype(np.uint8)

# ============================================================
# Main pipeline (chunked)
# ============================================================
def main():
    if any(m not in MODELS for m in ACTIVE_MODES):
        missing = [m for m in ACTIVE_MODES if m not in MODELS]
        raise ValueError(f"ACTIVE_MODES contains undefined modes: {missing}")

    disk_report("RUN START (pre-clean)")
    setup_environment()

    if CLEAN_RUN_CACHES:
        safe_rmtree(CACHE_ROOT)
        CACHE_ROOT.mkdir(parents=True, exist_ok=True)

    if CLEAN_NNUNET_PREPROCESSED:
        safe_rmtree(NNUNET_PREPROCESSED)

    if CLEAN_NNUNET_RESULTS:
        safe_rmtree(NNUNET_RESULTS)

    safe_mkdir_clean(FINAL_TIFF_DIR)
    safe_mkdir_clean(TEST_INPUT_DIR)
    for m in ACTIVE_MODES:
        safe_mkdir_clean(NNUNET_OUT[m])

    setup_environment()
    disk_report("AFTER CLEANUP")

    print("\n=== Registering ACTIVE_MODES ===")
    for m in ACTIVE_MODES:
        ensure_model_registered(MODELS[m])

    all_tifs = list_all_test_tifs()
    save_prob_flag = detect_save_prob_flag()

    print("\n=== FUSION CONFIG ===")
    print("ACTIVE_MODES:", ACTIVE_MODES)
    print("FUSION_SCHEME:", FUSION_SCHEME)
    w_norm = normalize_weights(ACTIVE_MODES, WEIGHTS)
    print("WEIGHTS (normalized over active):", {k: round(v, 6) for k, v in w_norm.items()})
    print("PROB_T:", PROB_T)
    print("PROCESS_IN_CHUNKS:", PROCESS_IN_CHUNKS, "| CASES_PER_CHUNK:", CASES_PER_CHUNK)

    if SUB_ZIP.exists():
        SUB_ZIP.unlink()

    # UPDATED pp_params for closing(7,7)+check holes
    pp_params = {
        'small_obj_thresh': 20000,
        'border_width': 3,
        'max_sheets': 40,
        'max_patch_size': 64,
        'close_par': 7,
        'close_perp': 7,
    }

    chunks = [all_tifs]
    if PROCESS_IN_CHUNKS:
        chunks = chunked(all_tifs, CASES_PER_CHUNK)

    disk_report("BEFORE ALL CHUNKS")

    with zipfile.ZipFile(SUB_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for ci, chunk_tifs in enumerate(chunks, start=1):
            print(f"\n==================== CHUNK {ci}/{len(chunks)} ====================")
            print(f"Chunk cases: {len(chunk_tifs)}")

            case_ids = prepare_test_data_subset(chunk_tifs)

            for m in ACTIVE_MODES:
                safe_mkdir_clean(NNUNET_OUT[m])

            disk_report(f"CHUNK {ci}: BEFORE PREDICT")
            run_active_models_in_stages(ACTIVE_MODES, save_prob_flag)
            disk_report(f"CHUNK {ci}: AFTER PREDICT")

            for cid in tqdm(case_ids, desc=f"Chunk {ci}: Fuse+PP+Zip"):
                p = fuse_probability_from_nnunet(cid, ACTIVE_MODES)
                mask = (p >= PROB_T).astype(np.uint8)
                pp_mask = postprocess(mask, pp_params)

                tif_path = FINAL_TIFF_DIR / f"{cid}.tif"
                tifffile.imwrite(tif_path, pp_mask)

                if ZIP_TIFF_IMMEDIATELY:
                    z.write(tif_path, tif_path.name)
                    tif_path.unlink()

                # delete per-case npz immediately
                for m in ACTIVE_MODES:
                    try:
                        npz = find_case_npz(NNUNET_OUT[m], cid)
                        npz.unlink()
                    except Exception:
                        pass

            for m in ACTIVE_MODES:
                safe_rmtree(NNUNET_OUT[m])
            safe_rmtree(TEST_INPUT_DIR)

            disk_report(f"CHUNK {ci}: AFTER CLEANUP")

        if not ZIP_TIFF_IMMEDIATELY:
            tiffs = sorted(FINAL_TIFF_DIR.glob("*.tif"))
            for tp in tqdm(tiffs, desc="Zipping (final pass)"):
                z.write(tp, tp.name)
                tp.unlink()

    n_test = len(all_tifs)
    with zipfile.ZipFile(SUB_ZIP, "r") as z:
        n_zip = sum(1 for n in z.namelist() if n.endswith(".tif"))
    print(f"\nSANITY: test_cases={n_test} | zip_tifs={n_zip}")

    safe_rmtree(FINAL_TIFF_DIR)
    safe_rmtree(TEST_INPUT_DIR)

    disk_report("RUN END")
    print("\nDONE.")
    print("submission:", SUB_ZIP, f"({SUB_ZIP.stat().st_size/1024/1024:.1f} MB)")

main()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.6.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.0 requires rmm-cu12==25.6.*, but you have rmm-cu12 25.2.0 which is incompatible.
WORKING_DIR (/tmp scratch): /tmp/vesuvius_multi_scratch
OUTPUT_DIR  (/kaggle/working): /kaggle/working
nnUNet_raw: /tmp/vesuvius_multi_scratch/nnUNet_data/nnUNet_raw
nnUNet_preprocessed: /tmp/vesuvius_multi_scratch/nnUNet_data/nnUNet_preprocessed
nnUNet_results: /tmp/vesuvius_multi_scratch/nnUNet_results
CACHE_ROOT: /tmp/vesuvius_multi_scratch/_cache
TMP_ROOT: /tmp/vesuvius_multi_scratch/_tmp
nnUNet_USE_BLOSC2: 1
CUDA_VISIBLE_DEVICES (parent): ALL
WORKING_DIR (/tmp scratch): /tmp/vesuvius_m

Preparing test data (chunk):   0%|          | 0/1 [00:00<?, ?it/s]


=== Stage 1: parallel inference ===
GPU0: nnUNetv2_predict -d 107 -c 3d_fullres -f all -i /tmp/vesuvius_multi_scratch/test_input -o /tmp/vesuvius_multi_scratch/nnunet_out_m7 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_4000epochs -npp 1 -nps 1 --verbose --save_probabilities
GPU1: nnUNetv2_predict -d 105 -c 3d_fullres -f all -i /tmp/vesuvius_multi_scratch/test_input -o /tmp/vesuvius_multi_scratch/nnunet_out_m5 -p nnUNetResEncUNetXLPlans -tr nnUNetTrainer_250epochs -npp 1 -nps 1 --verbose --save_probabilities
[M7_07_LPlans_4000_patch192_fromscratch_final] /usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type`

Chunk 1: Fuse+PP+Zip:   0%|          | 0/1 [00:00<?, ?it/s]


SANITY: test_cases=1 | zip_tifs=1

DONE.
submission: /kaggle/working/submission.zip (0.6 MB)
